
## Let's Build an EV Charging Network with optimization.
## Our task...
We are in charge of deploying EV charging stations for a municipal parking authority.
- Management would like to install new charging stations at a single site using the available resources from our current budget.
- Right now, we are asked to **maximize the total number of vehicles we can serve per day** from the chargers that we place at this site.
- The charging station types we can install are Level 1 (120V), Level 2 (240V), DC Fast 50kW, DC Fast 150kW, and DC Ultra-Fast 350kW.
- Each type of charger costs a different amount to install
- Therefore, we are constrained by our available **budget**.

## Initial questions
- What are our decisions?
- What are our constraints? 
- What is the objective?
- What data do we need right now?

## The Model
A lot of programming (in Python) is *imperative* -- just providing sequential instructions to complete. But mathematical optimization (aka math programming) is *declarative*. The math programming model does not tell the Gurobi solver what to do specifically. Instead, the model tells the Gurobi solver what the solution must look like. Gurobi then finds the solution in its own way.

Math optimization models take two forms:
- The **formulation**: An algebraic representation of the model (what we saw in our introduction!)
- The **code**: Writing the formulation in syntax to some software package.

In [ ]:
#%pip install pandas
#%pip install gurobipy
#%pip install plotly
#%pip install nbformat

import pandas as pd
import gurobipy as gp
from gurobipy import GRB
import plotly.express as px

Math programming always starts with the creation of a new model. We then add to it the *declarations* about the final solution. 


In [ ]:
model = gp.Model("EV Charging Network")

### The Variables

The model needs something to receive the solution values that the solver finds. It declares *variables* to hold those solution values. The mathematical optimization model **never explicitly sets these values**. It just describes them and makes rules about the values they can hold. This model declares a variable for each type of charging station.

***Let $x_i$ be the number of charging stations installed of type $i \in \{\text{Level1, Level2, Fast50, Fast150, Ultra350}\}$***

First, let's create data for each charging station type and its characteristics.

In [ ]:
chargers = ["Level1", "Level2", "Fast50", "Fast150", "Ultra350"]
data = pd.Series([3, 12, 30, 55, 80], 
                  index=chargers, 
                  name='vehicles_per_day')
cost_data = pd.Series([1200, 6500, 45000, 110000, 190000], 
                  index=chargers, 
                  name='cost')
data

#### Anyone know another way to hardcode this another way?... using a `gurobipy` object?
**Multidict** function splits one dictionary into multiple, allowing for easier access to data while model building

In [ ]:
chargers, vehicles_per_day, cost = gp.multidict(
    {
        "Level1":     [3, 1200],
        "Level2":     [12, 6500],
        "Fast50":    [30, 45000],
        "Fast150":   [55, 110000],
        "Ultra350":  [80, 190000]
    }
)


In [ ]:
print(chargers)

In [ ]:
print(vehicles_per_day)
print(cost)

Next, variables are added to the model -- one for each charger type. The main types of decision variables are
- CONTINUOUS
- INTEGER
- BINARY

#### What type of decision variable should we use in this case?
- Our decisions are the *number of charging stations* we should install for each type

In [ ]:
x = model.addVars(chargers, vtype = GRB.INTEGER, name = "chargers")
model.update()
x

Notice how these variables don't have values yet - the value of these variables is what Gurobi will decide when we optimize. Now let's define what we're looking to achieve, and what rules these values must abide by. 

### The Objective Function:

The math model must describe an objective also using algebra. When the model is solved, the value of **the objective function will be the maximum or minimum possible while following the rules described in the constraints**.

This model will **maximize the total number of vehicles served per day**. The objective function multiplies the quantity of each charger type installed times its daily vehicle throughput for all charger types.
For example, the total vehicles served by Level 2 chargers is $13*x_{Level2}$.

So the total vehicles served per day is

\begin{equation*}
3*x_{Level1} + 12*x_{Level2} + 30*x_{Fast50} + ... + 80*x_{Ultra350}
\end{equation*}

In [ ]:
### Option 1 - Written term by term:
model.setObjective(x["Level1"] * vehicles_per_day["Level1"] + 
                   x["Level2"] * vehicles_per_day["Level2"] + 
                   x["Fast50"] * vehicles_per_day["Fast50"] + 
                   x["Fast150"] * vehicles_per_day["Fast150"] + 
                   x["Ultra350"] * vehicles_per_day["Ultra350"], 
                    sense=GRB.MAXIMIZE)

Let's use a bit more math notation. 

Let $v_i$ be the vehicles served per day by charger type $i$
\begin{equation*}
  \text{Maximize} \space \sum_i v_i*x_i
\end{equation*}

In [ ]:
### Option 2 - Here is another way to write the same thing:
model.setObjective(gp.quicksum(x[i] * vehicles_per_day[i] for i in chargers), sense=GRB.MAXIMIZE)

### Option 3 - The prod function is a handy function provided by the Gurobi API to further simplify our code.
model.setObjective(x.prod(vehicles_per_day), sense=GRB.MAXIMIZE)

### The Constraints:

This parking authority has **limited resources** available to deploy the charging stations. The model must make sure to only install chargers for which the resources are available.

Here is a list of resources and how much is available:

In [ ]:
resources, available = gp.multidict(
    {
        "budget":     350000,  # dollars
    }
)

In [ ]:
print(resources)
print(available)

Here is how much of each resource is required for each charger type. 

In [ ]:
print(cost)

Let's introduce **constraints** to make sure we don't use more resources than we have available. **Constraints are where the *rules* acting on decision variables are declared.**

For example, the amount of total cost of installing all chargers must be less than or equal to the available budget.

\begin{equation*}
\text{total spend} \le 350000
\end{equation*}

Let's write an expression for the total spend used using our decision variable, $x_{cost}$.
\begin{align*}
\text{total cost used} = \space& cost_{Level1}* x_{Level1} + cost_{Level2}*x_{Level2} + ... + cost_{Ultra350}*x_{Ultra350} \\
& cost_{Level1}* x_{Level1} + cost_{Level2}*x_{Level2} + ... + cost_{Ultra350}*x_{Ultra350} \le 350000
\end{align*}

In [ ]:
### Option 1 - A very explicit way to write these constraints:

# total budget <= budget available
model.addConstr(x["Level1"] * cost["Level1"] + x["Level2"] * cost["Level2"]
                + x["Fast50"] * cost["Fast50"] + x["Fast150"] * cost["Fast150"]
                + x["Ultra350"] * cost["Ultra350"] <= available["budget"], "budget limit")

### The Solution:
It's as simple as one line of code to run the optimization, then we query the decision variables for their values (assuming the optimization completed successfully)

In [ ]:
model.optimize()

In [ ]:
for v in model.getVars():
    print(f'{v.varName}: {v.x}')
print(f'Obj: {model.objVal}')

### More realism:

But what if our problem becomes a bit more realistic? What if we need to respect more than just the budget limitations - things like power and physical space?

This parking authority has **limited resources** beyond just budget available to deploy the charging stations. We must make sure to respect all restrictions.

Here is we will just add a couple, but this could be an even longer list:

In [ ]:
resources, available = gp.multidict(
    {
        "budget":     350000,  # dollars
        "power":        400,  # kW
        "space":         20,  # square meters
    }
)

In [ ]:
print(resources)
print(available)

Here is how much of each resource is required for each charger type. 

In [ ]:
chargers, vehicles_per_day, budget, power, space = gp.multidict({
    "Level1":   [3, 1200,  1.4,   1],
    "Level2":   [12, 6500,  11,   1],
    "Fast50":   [30, 45000,  50.0,  2],
    "Fast150":  [55, 110000, 150.0, 3],
    "Ultra350": [80, 190000, 350.0, 4],
})

requirements = {"budget": budget, "power": power, "space": space}
requirements

In [ ]:
### Option 1 - A very explicit way to write these constraints, same as in our simple example:

# total budget <= budget available
model.addConstr(x["Level1"] * requirements["budget"]["Level1"] + x["Level2"] * requirements["budget"]["Level2"]
                + x["Fast50"] * requirements["budget"]["Fast50"] + x["Fast150"] * requirements["budget"]["Fast150"]
                + x["Ultra350"] * requirements["budget"]["Ultra350"] <= available["budget"], "budget limit")

# total power <= power available
model.addConstr(x["Level1"] * requirements["power"]["Level1"] + x["Level2"] * requirements["power"]["Level2"]
                + x["Fast50"] * requirements["power"]["Fast50"] + x["Fast150"] * requirements["power"]["Fast150"]
                + x["Ultra350"] * requirements["power"]["Ultra350"] <= available["power"], "power limit")

# total space <= space available
model.addConstr(x["Level1"] * requirements["space"]["Level1"] + x["Level2"] * requirements["space"]["Level2"]
                + x["Fast50"] * requirements["space"]["Fast50"] + x["Fast150"] * requirements["space"]["Fast150"]
                + x["Ultra350"] * requirements["space"]["Ultra350"] <= available["space"], "space limit")

We can also generalize the *quantity* of each resource with $q_j$ using a bit more math notation:
\begin{equation*}
  \sum_{i}r_{i, j} * x_{i} \le q_j, \space \text{for all} \space j \space \text{in} \space \{\text{budget, power, space}\}
\end{equation*}

Note: A short way to write "for all" is using the symbol $\forall$. Also, $\in$ means "in", or "an element of." We saw this notation in the intro.

Where the following can be found using requirements[('Level2', 'power')]
\begin{equation*} r_{Level2, power} \end{equation*}

We can change the code so it looks like this condensed notation and loop through each resource using `quicksum`:

In [ ]:
### Option 2 - Using quicksum
for resource in resources:
    model.addConstr(gp.quicksum(x[i] * requirements[resource][i] for i in chargers) <= available[resource], name="resource usage")

Even more compact, and makes it easier to store the constraints as an object:

In [ ]:
### Option 3 - Using quicksum and storing constraint as an object
resource_constraints = model.addConstrs((gp.quicksum(x[i] * requirements[j][i] for i in chargers) <= available[j] for j in resources), name="resource usage")

### The Updated Solution:

In [ ]:
model.optimize()

As we can see, this model runs incredibly fast because it's quite small and simple to solve. 

Let's look at the solution that Gurobi found.

In [ ]:
# use `VarName` and `X` to get t]he variables name and value, respectively:
for v in model.getVars():
    print(f"{v.VarName}: {v.X}")

## Congratulations!
You just ran an optimization model!

# Let's make this a step more realistic
Where we left off, a heuristic or possibly greedy approach would have worked to solve that model. But problems typically have much more complicated business rules to follow, and are larger in scale-- which highlights the strengths of mathematical optimization.

## Before we continue - Let's Create a function to make solving and reporting a bit more straightforward
This function will optimize the model and print the solution as we did in the last example.

In [ ]:
def solve_and_print_solution(model, x, chargers, sites):
    # Optimize model
    model.setParam("outputflag", 0)
    model.optimize()
    
    # Check the model status
    if model.status == GRB.OPTIMAL:
        print("Optimal solution found\n")
        print('------------------')
        # Print objective value
        print(f"Objective value: {model.objVal}\n")
        print('------------------')
        # Print site-level results
        for s in sites:
            print(f"{s}:")

            # Total chargers at this site
            total_at_site = sum(x[s, c].X for c in chargers)

            if total_at_site == 0:
                print("  No. chargers allocated at this site\n")
            else:
                for c in chargers:
                    if x[s, c].X > 0:
                        print(f"  {c}: {x[s, c].X}")
                print()  # blank line for readability

        # --- Metrics (independent of objective) ---

        total_cost = sum(
            chargers_data.installation_cost[j] * x[s, j].X
            for s in sites
            for j in chargers
        )

        total_vehicles = sum(
            chargers_data.vehicles_per_day[j] * x[s, j].X
            for s in sites
            for j in chargers
        )

        total_chargers = sum(
            x[s, j].X
            for s in sites
            for j in chargers
        )

        # --- Print ---
        print(f"Objective Value: {model.objVal:,.0f}")
        print(f"Total Cost: {total_cost:,.0f}")
        print(f"Total Vehicles Served: {total_vehicles:,.0f}")
        print(f"Total Chargers Installed: {total_chargers:,.0f}\n")

    else:
        print(f"Model status: {model.status}")
        #sys.exit(1)

## Let's solve a more realistic business problem 
When setting up the problem the first time we used `multidict` to create the input data for our model. This time, let's read data in from a csv.

If you are on colab, use this cell.

In [ ]:
# ### This time, we read in data from csv
# path = 'https://raw.githubusercontent.com/spurschke98/Gurobi-EV-Charging-Example/main/'
# global_available = pd.read_csv(path+'data_files/global_resources_available.csv', index_col=['resource']).squeeze() # Global resources - Equipment and Budget
# site_info = pd.read_csv(path+'data_files/sites.csv', index_col=['site']) # Site resources - Power and Space
# chargers_data = pd.read_csv(path+'data_files/chargers_data.csv', index_col=['chargers']).squeeze()
# requirements = pd.read_csv(path+'data_files/requirements.csv', index_col=['chargers', 'resources']).squeeze()
# sites_data = pd.read_csv(path+"data_files/sites.csv")
# global_resources = global_available.index.to_list()
# site_resources = ['power', 'space']
# chargers = chargers_data.index.to_list()
# sites = site_info.index.to_list()
# site_available_resources = site_info[['power', 'space']].stack().to_dict()

If you downloaded the files locally, run this cell:

In [ ]:
### This time, we read in data from csv
global_available = pd.read_csv('data_files/global_resources_available.csv', index_col=['resource']).squeeze() # Global resources - Equipment and Budget
site_info = pd.read_csv('data_files/sites.csv', index_col=['site']) # Site resources - Power and Space
chargers_data = pd.read_csv('data_files/chargers_data.csv', index_col=['chargers']).squeeze()
requirements = pd.read_csv('data_files/requirements.csv', index_col=['chargers', 'resources']).squeeze()
sites_data = pd.read_csv("data_files/sites.csv")
global_resources = global_available.index.to_list()
site_resources = ['power', 'space']
chargers = chargers_data.index.to_list()
sites = site_info.index.to_list()
site_available_resources = site_info[['power', 'space']].stack().to_dict()

In [ ]:
print("total demand: ",site_info.demand.sum())
site_info

## Create the model 
While the original model was only concerned with one site, a realistic business problem that necessitates mathematical optimization is large in scale. For this use case, that means **multiple candidate sites**. 

- Management would like to install charging stations across these sites while constrained by limited resources, including total budget, electrical capacity, physical space, and equipment units.
- Each site has different characteristics and constraints (e.g., varying available space or grid capacity), but all installations draw from the same budget and equipment pool. 
- Our goal is to decide how many chargers of each type to install **at each site**.

For this model: 
- $x_{t,i}$ is the number of charging stations to install of type $i \in \{\text{Level1, Level2, Fast50, Fast150, Ultra350}\}$ at site $t \in T$.
- $v_i$ is the vehicles served per day by each charger type (assumed to be the same across all sites).
- $q_j$ is the total amount of resource $j \in \{\text{budget, equipment}\}$ available across all sites.
- $q_{t,j}$ is the amount of resource $j\in \{\text{power,space}\}$ available at site $t$.
- $r_{i,j}$ is the amount of resource $j$ required to install charger type $i$.
- t is a site in the list of sites, T 

\begin{align*}
\text{Maximize} \quad & \sum_{t \in T} \sum_{i \in I} v_i \cdot x_{t,i} \\

\text{subject to:} \quad & \\

& \sum_{t \in T} \sum_{i \in I} r_{i,j} \cdot x_{t,i} \le q_j, 
\quad \forall j \in \{\text{budget, equipment}\} \\

& \sum_{i \in I} r_{i,j} \cdot x_{t,i} \le q_{t,j}, 
\quad \forall t \in T,\; j \in \{\text{power, space}\} \\

& x_{t,i} \ge 0, \quad \forall t \in T,\; \forall i \in \text{Chargers}
\end{align*}

The change in this formulation is adding a dimension since we have multiple sites.  This drastically increases the number of combinations one would need to consider to land on the optimal solution.  Luckily, Gurobi algorithmically solves the problem, avoiding calculating and comparing every possible combination.

Let's look at where the possible sites are around the Boston Area

In [ ]:
# Create map
fig = px.scatter_map(
    sites_data,
    lat="latitude",
    lon="longitude",
    hover_name="site",
    hover_data={"power": True, "space": True},
    zoom=12,
    height=600
)

fig.show()

**Initalize our model**

In [ ]:
### Create a new model - this will replace the original 
model = gp.Model("EV Charging Network - Realistic")

**Decision Variables**

For this model, we need a decision variable for each charger, ***at each site***

In [ ]:
### Decision variables
x = model.addVars(sites, chargers, vtype=GRB.INTEGER, name="charger_assignments")
model.update()
x


**Constraints**

We now have resources that are shared across all sites (budget, equipment), and site-level resource constraints (power, space). 

In [ ]:
global_available

In [ ]:
### Global resource constraints (shared across all sites)
global_resource_constraints = model.addConstrs(
    # The total resource utilization across all sites must not exceed the global available resources (budget and equipment)
    (gp.quicksum(x[t, i] * requirements[i, j] for t in sites for i in chargers)
        <= global_available[j]
        for j in global_available.keys()
    ),
    name="global_resource_usage"
)

In [ ]:
### Site-level constraints for power and physical space 
site_resource_constraints = model.addConstrs(
    (
        gp.quicksum(x[t, i] * requirements[i, j] for i in chargers)
        <= site_available_resources[t, j]
        for t in sites
        for j in site_resources
    ),
    name="site_resource_usage"
)

**Objective Function**

Our goal is to maximize the number of vehicles served per day across all sites. 

In [ ]:
### Objective function
model.setObjective(
    gp.quicksum(
        x[t, i] * chargers_data.vehicles_per_day[i]
        for t in sites
        for i in chargers
    ),
    sense=GRB.MAXIMIZE
)

### Optimize

In [ ]:
### Optimize

solve_and_print_solution(model, x, chargers, sites)

## Decision expressions
Sometimes it is helpful to store quantities of interest to help make code easier to read, reduce clutter, or get key values quickly.

Suppose we have grouped charger types into two categories: standard and fast. Fast chargers are Fast50, Fast150, and Ultra350, with the rest being standard.

Code the expressions below containing decision variables.

Let's define some sets.
- $I = \{\text{Level1, Level2, Fast50, Fast150, Ultra350}\}$
- $F = \{\text{Fast50, Fast150, Ultra350}\}$
- $S = I - F$


\begin{align*}
\text{total chargers installed} & = \sum_{i \in I} x_i \\
\text{vehicles served by all chargers} & = \sum_{t \in T} \sum_{i \in I} v_{t,i} \cdot x_{t,i} \\
\text{vehicles served by fast chargers} & = \sum_{t \in T} \sum_{f \in F} v_{t,f} \cdot x_{t,f} \\
\text{vehicles served by standard chargers} & = \sum_{t \in T} \sum_{s \in S} v_{t,s} \cdot x_{t,s}
\end{align*}

In [ ]:
fast_chargers = ['Fast50', 'Fast150', 'Ultra350']
standard_chargers = [i for i in chargers if i not in fast_chargers]



In [ ]:
## How many chargers are installed?
# Total?
total_chargers = gp.quicksum(x[t, i] for t in sites for i in chargers)
# At each site?
total_chargers_per_site = {
    t: gp.quicksum(x[t, i] for i in chargers)
    for t in sites
}

## How many Fast Chargers are installed?
# Total?
total_fast_chargers = gp.quicksum(x[t, f] for t in sites for f in fast_chargers)
# At each site?
total_fast_chargers_per_site = {
    t: gp.quicksum(x[t, f] for f in fast_chargers)
    for t in sites
}

## How many Standard Chargers are installed?
# Total?
total_standard_chargers = gp.quicksum(x[t, s] for t in sites for s in standard_chargers)
# At each site?
total_standard_chargers_per_site = {
    t: gp.quicksum(x[t, s] for s in standard_chargers)
    for t in sites
}

## How many Vehicles are we serving per day with all the chargers?
# Total?
vehicles_all = gp.quicksum(
    chargers_data.vehicles_per_day[i] * x[t, i]
    for t in sites
    for i in chargers
)
# At each site?
vehicles_per_site = {
    t: gp.quicksum(
        chargers_data.vehicles_per_day[i] * x[t, i]
        for i in chargers
    )
    for t in sites
}

## How many Vehicles are we serving per day with all Fast chargers?
# Total?
vehicles_all_fast = gp.quicksum(
    chargers_data.vehicles_per_day[f] * x[t, f]
    for t in sites
    for f in fast_chargers
)
# At each site?
vehicles_per_site_fast = {
    t: gp.quicksum(
        chargers_data.vehicles_per_day[f] * x[t, f]
        for f in fast_chargers
    )
    for t in sites
}

## How many Vehicles are we serving per day with all Standard chargers?
# Total?
vehicles_all_standard = gp.quicksum(
    chargers_data.vehicles_per_day[s] * x[t, s]
    for t in sites
    for s in standard_chargers
)
# At each site?
vehicles_per_site_standard = {
    t: gp.quicksum(
        chargers_data.vehicles_per_day[s] * x[t, s]
        for s in standard_chargers
    )
    for t in sites
}

In [ ]:
# Update model object to apply changes
model.update()

In [ ]:
for site in sites:
    print(site, ": ", vehicles_per_site[site].getValue(), " vehicles served")

## Changing our model
One of the key strengths of mathematical optimization is its flexibility. When requirements or business rules change, you don’t need to start over or switch methods, you simply update the model.

By adding or adjusting constraints, variables, or objectives, you can easily reflect new conditions. 

### Updated objective and new constraint
What if instead of the objective to be serving as many vehicles as possible, we made this a requirement (or a constraint), and instead made our objective to satisfy all demand for as cheap as possible?

$$
\sum_{i \in I} v_i \cdot x_{t,i} \ge d_t \quad \forall t \in T
$$

In [ ]:
demand_fulfillment = model.addConstrs(
    (
        gp.quicksum(
            # Vehicles per day for each charger * # of that type of charger at that site (variable) >= demand 
            chargers_data.vehicles_per_day[i] * x[t, i]
            for i in chargers
        )
        >= site_info['demand'][t]
        for t in sites
    ),
    name="demand_satisfaction"
)

solve_and_print_solution(model, x, chargers,sites)

In [ ]:
#for site in sites:
#    print(site, ": ", vehicles_per_site[site].getValue(), " vehicles served")

### New objective function - Installation Cost
Instead of maximizing the vehicles served, we are asked to minimize the installation costs, while respecting demand.

\begin{equation*}
\min \quad \sum_{t \in T} \sum_{i \in I} c_i \cdot x_{t,i}
\end{equation*}

In [ ]:
model.setObjective(
    gp.quicksum(
        chargers_data.installation_cost[i] * x[t, i]
        for t in sites
        for i in chargers
    ),
    sense=GRB.MINIMIZE
)
solve_and_print_solution(model, x, chargers,sites)

In [ ]:
for site in sites:
    print(site, ": demand of ", site_info['demand'][site], " vehicles, ", vehicles_per_site[site].getValue(), " vehicles served")

Note that we didn't have to do anything else other than re-run `setObjective` !

In [ ]:
print('Vehicles served by fast chargers across all sites:', vehicles_all_fast.getValue())
print('Vehicles served by standard chargers across all sites:', vehicles_all_standard.getValue())

In [ ]:
vehicles_all_fast.getValue() / vehicles_all.getValue()

### Service Balance Between Standard and Fast Chargers
The vehicles served by fast chargers can be no more than x% of the total vehicles served.
*Reminder: our objective function is still minimizing cost*

\begin{equation*}
\sum_{t \in T} \sum_{s \in F} v_f \cdot x_{t,f} \le xpercent \cdot \sum_{t \in T} \sum_{i \in I} v_i \cdot x_{t,i}
\end{equation*}

In [ ]:
### The vehicles served by fast chargers must be no more than x% of all vehicles served
## can be changed based on what value would be nonbinding
val = 0.6
service_balance = model.addConstr(vehicles_all_fast <= val * vehicles_all, name='service_balance')

solve_and_print_solution(model, x, chargers,sites)

In [ ]:
print('Vehicles served by fast chargers across all sites:', vehicles_all_fast.getValue())
print('Vehicles served by standard chargers across all sites:', vehicles_all_standard.getValue())

In [ ]:
vehicles_all_fast.getValue() / vehicles_all.getValue()

**Discussion**: The constraint didn't change anything! Why?
- The constraint is already satisfied, so it's a **non-binding constraint**. Adding this constraint does not change the optimal solution.

**Key Learning**: Not all constraints are active in every solution! Some constraints only matter when certain decisions are made.

**To see this constraint in action**, we need a scenario where standard chargers are more incentivized. Let's switch back to maximizing vehicles served:

In [ ]:
### Change objective back to maximize vehicles served
model.setObjective(gp.quicksum(requirements[i, 'power'] * x[t, i] for t in sites for i in chargers), GRB.MINIMIZE)
solve_and_print_solution(model, x, chargers,sites)

In [ ]:
print('Vehicles served by fast chargers across all sites:', vehicles_all_fast.getValue())
print('Vehicles served by standard chargers across all sites:', vehicles_all_standard.getValue())

In [ ]:
vehicles_all_fast.getValue() / vehicles_all.getValue()

**Now the constraint matters!** When maximizing vehicles served, we want fast chargers, but the service balance constraint limits how many we can install.

We stored each constraint as an object, which makes it easier to interact with these later. Now, let's remove the `service_balance` constraint.

In [ ]:
model.remove(service_balance)
model.update()
model.optimize()

### Slack and Binding Constraints

The `Slack` of a constraint is the gap between the left-hand side and right-hand side values at the optimal solution. You can also think of this as how far the value is from its bound, in our case upper bound. Printing this for the `resource_usage` constraints will show how much of each resource is remaining.

In [ ]:
for j in global_resources:
    print(f"Remaining {j}: {global_resource_constraints[j].Slack}")

**Question**: Which one of the resource constraints is binding? Can a constraint be binding if the remaining resource value is not 0?

### Let's check in on our model and write it to an LP file
As you build your model, it’s helpful to periodically verify that variables, constraints, and the objective are being added as intended. Writing the model to a file lets you inspect exactly what’s been created and catch issues early.

LP and MPS are standard file formats for exporting optimization models, making them easy to review, debug, and share.

In [ ]:
### Writing a *.lp file is a great way to get a look at your model
model.write('our_model.lp')
model.write('our_model.mps')

### Meet minimum service requirements

Management has some operational requirements:
- We must have **at least 15 fast chargers** (any combination of Fast50, Fast150, or Ultra350) for customers who need quick charging
- We must have **at least 25 standard chargers** (any combination of Level1 or Level2) for customers with more time

Recall that $F = \{Fast50, Fast150, Ultra350\}$ and $S = \{Level1, Level2\}$.  Let \(T\) be the set of sites. We can model these minimum service requirements as:

\begin{align*}
\sum_{t \in T} \sum_{f \in F} x_{t,f} &\ge 15 \quad \text{(minimum fast chargers)} \\
\sum_{t \in T} \sum_{s \in S} x_{t,s} &\ge 25 \quad \text{(minimum standard chargers)}
\end{align*}

In [ ]:
print('Number of fast chargers installed across all sites:', total_fast_chargers.getValue())
print('Number of standard chargers installed across all sites:', total_standard_chargers.getValue())


In [ ]:

# At least 15 fast chargers across all sites
min_fast = model.addConstr(
    total_fast_chargers >= 15,
    name="min_fast"
) 
# At least 25 standard chargers across all sites
min_standard = model.addConstr(
    total_standard_chargers >= 25,
    name="min_standard"
)
solve_and_print_solution(model, x, chargers, sites)

### Cost of installing chargers:
- In the data frame above, we see there is an installation cost per charger. 
- Now management asks: **What's the cheapest way to serve at least 1000 vehicles per day?**
- Write an expression for the total *variable cost* of chargers installed.
- Add a constraint that we must serve at least 1000 vehicles per day.
- Make that the new objective to minimize total cost and solve.

Let $c_i$ be the installation cost of charger type $i$, and let $T$ be the set of sites.

$$
\text{Minimize} \quad \sum_{t \in T} \sum_{i \in I} c_i \cdot x_{t,i} \quad \text{(Total Installation Cost of Chargers)}
$$

$$
\text{subject to:} \quad \sum_{t \in T} \sum_{i \in I} v_i \cdot x_{t,i} \ge 1000 
$$

In [ ]:
### Define the total Variable installation cost
vCost_all_chargers = gp.quicksum(
    chargers_data.installation_cost[i] * x[t, i]
    for t in sites
    for i in chargers
)

In [ ]:
### Add service level constraint - must serve at least Vmin vehicles per day
min_service_level = 1000

service_requirement = model.addConstr(
    vehicles_all >= min_service_level,
    name="service_requirement"
)

model.setObjective(vCost_all_chargers, GRB.MINIMIZE)

solve_and_print_solution(model, x, chargers, sites)


What happens if we increase the minimum service level? 

In [ ]:
model.remove(service_requirement)

### Add service level constraint - must serve at least Vmin vehicles per day
min_service_level = 3000

service_requirement = model.addConstr(
    vehicles_all >= min_service_level,
    name="service_requirement"
)

model.setObjective(vCost_all_chargers, GRB.MINIMIZE)

solve_and_print_solution(model, x, chargers, sites)


What does this tell us?

Model Status: 3 infers that our model is infeasible. An infeasible model means that there is no solution that exists that satisfies all constraints. While it is possible to service 1000 vehicles with the limited resources that we have to install chargers, it is not possible to service 3000 vehicles.

For more on diagnosing and repairing infeasibility, see this article: https://support.gurobi.com/hc/en-us/articles/360029969391-How-do-I-determine-why-my-model-is-infeasible. 

In [ ]:
model.remove(service_requirement)
model.update()

solve_and_print_solution(model, x, chargers, sites)


### Fixed costs: More decision variables
Our installation crew can only install one type of charger at a time. When we want to switch to a different type, we incur permitting and setup costs, which adds a fixed cost to each charger type. Given this, we have to decide whether or not we want to install each type of charger.

**Key question**: Will adding fixed costs change which charger types we choose when trying to minimize cost while serving 500 vehicles/day?

Let's go step-by-step to add in the fixed costs for each charger type.
- First define a new set of variables indexed by charger type and call them `y`.
- Write a constraint that links the number of chargers installed with the new binary variable.
- Don't resolve yet.

Let $f_i$ be the fixed cost of charger type $i$ and $y_i = 1$ if charger type $i$ is installed, $0$ otherwise.

If we decide to install a Fast50 charger, then $x_{Fast50} > 0$ and we want to have the associated cost go from 0 to the fixed cost.

So the constraint is
$$
x_{t,i} \le M_{t,i} \cdot y_{t,i} \quad \forall t \in T,\; i \in I
$$

But what do we choose for $M$?

In [ ]:
print("=== Current Solution (without fixed costs) ===")

current_cost = vCost_all_chargers.getValue()
current_vehicles = vehicles_all.getValue()

print(f"Total cost: ${current_cost:,.0f}")
print(f"Total Vehicles served: {current_vehicles:.0f}")

print("\nCharger types installed by site:")
for s in sites:
    print(f"{s}:")
    for i in chargers:
        if x[s, i].X > 0.5:
            cost = chargers_data.installation_cost[i] * x[s, i].X
            print(f"  {i}: {x[s, i].X:.0f} units (cost: ${cost:,.0f})")
    print()

In [ ]:
### Create a binary variable associated with each charger, at each site. 
y = model.addVars(sites, chargers, vtype=GRB.BINARY, name="install")

### Link binary variable to charger installation using `big-M` constraints
M = 100

# If y[t,i] = 0 → x[t,i] = 0
link_x_y_upper = model.addConstrs(
    (x[t, i] <= M * y[t, i] for t in sites for i in chargers),
    name="link_upper"
)

# If x[t,i] > 0 → y[t,i] = 1
link_x_y_lower = model.addConstrs(
    (x[t, i] >= y[t, i] for t in sites for i in chargers),
    name="link_lower"
)

model.update()

### Fixed costs: Combining costs
- Write an expression that finds the total fixed costs using the new variable and the `fixed_cost` column from `chargers_data`.
- Add that to the installation cost expression and set that as the new objective
- Solve!

**Question**: Do you think adding fixed costs will cause us to use fewer charger types?

$$
\text{Total fixed cost} = \sum_{t \in T} \sum_{i \in I} f_i \cdot y_{t,i}
$$

$$
\text{Total cost} = \text{total fixed cost} + \text{total installation cost} = \sum_{t \in T} \sum_{i \in I} \left( f_i \cdot y_{t,i} + c_i \cdot x_{t,i} \right)
$$

In [ ]:
# Total fixed cost
fixed_cost_total = gp.quicksum(
    chargers_data.fixed_cost[i] * y[t, i]
    for t in sites
    for i in chargers
)

# Total installation cost (you already had this)
installation_cost_total = gp.quicksum(
    chargers_data.installation_cost[i] * x[t, i]
    for t in sites
    for i in chargers
)

# Combined objective
total_cost = fixed_cost_total + installation_cost_total

model.setObjective(total_cost, GRB.MINIMIZE)
solve_and_print_solution(model, x, chargers, sites)


## Multi-objective optimization
- We are asked to maximize the total vehicles served while minimizing costs.
- Math optimization cannot **simultaneously** work on two objectives -- there's always a tradeoff.
- Two types of multi-objective: hierarchical, and weighted (blended).
- After talking more about how we want to prioritize the objectives, we want to:
    - First maximize the **vehicles served per day**.
    - Then minimize total costs
- Also include the following constraints we've already included
    - Resource limits
    - Meet minimum requirements
- gurobipy let's you do this quite easily! 
    - You can reference the documentation for more information: https://docs.gurobi.com/projects/optimizer/en/current/features/multiobjective.html


In [ ]:
model.ModelSense = GRB.MINIMIZE

# Objective 1 (highest priority): maximize vehicles served
model.setObjectiveN(
    vehicles_all,
    index=0,
    priority=2,
    weight=-1,
    name="maximize_vehicles"
)

# Objective 2 (lower priority): minimize cost
model.setObjectiveN(
    total_cost,
    index=1,
    weight=1,
    priority=1,
    name="minimize_cost"
)

solve_and_print_solution(model, x, chargers, sites)

### Logical constraints with binary variables
Now that we have binary decision variables that show which charger types are installed, we can model logical relationships between them. Model the following statements and write `gurobipy` code.

We won't solve a model with these.
- We **can** install either Fast150 **or** Ultra350 chargers, and **possibly both** (at least one).
- We **must** install either Fast150 **or** Ultra350 chargers, but **not both** (exactly one).
- We install between 2 and 3 types of standard chargers.

- We **can** install either Fast150 **or** Ultra350 chargers, and **possibly both** (at least one).

$$
y_{t,\text{Fast150}} + y_{t,\text{Ultra350}} \ge 1 \quad \forall t \in T
$$

- We **must** install either Fast150 **or** Ultra350 chargers, but **not both** (exactly one).

$$
y_{t,\text{Fast150}} + y_{t,\text{Ultra350}} = 1 \quad \forall t \in T
$$


- We install between 2 and 3 types of standard chargers.

$$
2 \le \sum_{i \in S} y_{t,i} \le 3 \quad \forall t \in T
$$

In [ ]:
# We can install either Fast150 or Ultra350 chargers, and possibly both (at least one)
model.addConstrs(
    (y[t, 'Fast150'] + y[t, 'Ultra350'] >= 1 for t in sites),
    name="at_least_one_fast"
)

# We must install either Fast150 or Ultra350 chargers, but not both (exactly one)
model.addConstrs(
    (y[t, 'Fast150'] + y[t, 'Ultra350'] == 1 for t in sites),
    name="exactly_one_fast"
)

# We install between 2 and 3 types of standard chargers (per site)
model.addConstrs(
    (gp.quicksum(y[t, i] for i in standard_chargers) >= 2 for t in sites),
    name="min_standard_types"
)

model.addConstrs(
    (gp.quicksum(y[t, i] for i in standard_chargers) <= 3 for t in sites),
    name="max_standard_types"
)

### Conditional statements
- If we install Level1 chargers, then we **must** install Level2 chargers.
- If we install Level1 chargers, then we **must** install Level2 **and** Fast50 chargers.
- If we install Level1 chargers, then we **must** install Level2 **or** Fast50 chargers (or both).

- If we install Level1 chargers, then we must install Level2 chargers
$$
y_{t,\text{Level1}} \le y_{t,\text{Level2}} \quad \forall t \in T
$$

- If we install Level1 chargers, then we must install Level2 and Fast50 chargers
$$
y_{t,\text{Level1}} \le y_{t,\text{Level2}} \quad \forall t \in T
$$

$$
y_{t,\text{Level1}} \le y_{t,\text{Fast50}} \quad \forall t \in T
$$

$$
2 \cdot y_{t,\text{Level1}} \le y_{t,\text{Level2}} + y_{t,\text{Fast50}} \quad \forall t \in T
$$

- If we install Level1 chargers, then we must install Level2 or Fast50 chargers
$$
y_{t,\text{Level1}} \le y_{t,\text{Level2}} + y_{t,\text{Fast50}} \quad \forall t \in T
$$

In [ ]:
#### Logical constraint candidates:

### If we install Level1 chargers, then we must install Level2 chargers
model.addConstrs(
    (y[t, 'Level1'] <= y[t, 'Level2'] for t in sites),
    name="level1_implies_level2"
)

### If we install Level1 chargers, then we must install Level2 and Fast50 chargers
model.addConstrs(
    (y[t, 'Level1'] <= y[t, 'Level2'] for t in sites),
    name="level1_implies_level2"
)

model.addConstrs(
    (y[t, 'Level1'] <= y[t, 'Fast50'] for t in sites),
    name="level1_implies_fast50"
)

model.addConstrs(
    (2 * y[t, 'Level1'] <= y[t, 'Level2'] + y[t, 'Fast50'] for t in sites),
    name="level1_implies_both"
)

### If we install Level1 chargers, then we must install Level2 or Fast50 chargers
model.addConstrs(
    (y[t, 'Level1'] <= y[t, 'Level2'] + y[t, 'Fast50'] for t in sites),
    name="level1_implies_either"
)

#### Dipose of the gurobipy environment

In [ ]:
model.dispose()